# The WSP Heuristic

Here we calculate the WSPD for a given problem and then check whether the number of edges crossing the each pair of clusters is the correct number given the number of exits out of the cluster as per the proved theorems. 

## Using internal python implementation of WSPD class

In [1]:
from pathlib import Path

import tsplib95
import numpy as np

from wsp import ds, tsp
from concurrent.futures import ThreadPoolExecutor

TREE_TYPE = ds.PKPRQuadTree
SIZE_THRESHOLD = 250
TSP_LIB_PATH = Path("TSPLIB")
S_FACTOR = 1.5
#ax = np.array([None, None])
#sys.setrecursionlimit(500_000)

In [2]:
#all_problems : dict[str, tsp.TravellingSalesmanProblem[TREE_TYPE]] = {}

#for file in sorted(TSP_LIB_PATH.iterdir()): # Loop through every tsp
#    if file.suffix != ".tsp" or not file.is_file():
#        continue
#    problem = tsplib95.load(f"{file.absolute()}")
#    if problem.edge_weight_type != "EUC_2D": # Skip non-Euclidean TSPs
#        continue
#    if problem.dimension > SIZE_THRESHOLD: # Skip large TSPs
#        continue

#    points = [ds.Point(*problem.node_coords[i]) for i in problem.get_nodes()]
#    ts_problem = tsp.TravellingSalesmanProblem[TREE_TYPE](TREE_TYPE, points, ax, s=S_FACTOR) 
    
#    all_problems[problem.name] = ts_problem
#    print(f"Added {problem.name}")

#print("Found", len(all_problems), "euclidean TSPs")

In [3]:
#for name, problem in all_problems.items():
#    tour, _, _ = problem.nnn_path
#    badCount = 0
#    for (A, B), _ in problem.wspd:
#        pointsA = set(A.covered_points)
#        pointsB = set(B.covered_points)
#        if not problem.wsp_heuristic_good(tour, pointsA, pointsB):
#            badCount += 1
#    if badCount > 0:
#        print(f"Problem {name} has {badCount} bad pairs")

## Faster Runner using external cpython libs
still solving problem as they are seen

no bad pairs found up to 12k for s=1.5  (TSPLIB EUC2D)
no bad pairs found up to 12k for s=1.0 (TSPLIB EUC2D)

In [4]:
%load_ext line_profiler
import itertools

import numba as nb
nb.config.THREADING_LAYER = 'tbb'
from numba import njit
from numba.typed import List as NumbaList
import elkai
from wspd import build_wspd_flat_np, point as wsp_point # uses diam based sep factor
import wspd

from utils.sort_by_key_inplace import sort_by_key_inplace

SIZE_THRESHOLD2 = 500

In [5]:
#all_problems : list[tsplib95.models.StandardProblem] = []

#for file in sorted(TSP_LIB_PATH.iterdir()): # Loop through every tsp
#    if file.suffix != ".tsp" or not file.is_file():
#        continue
#    problem = tsplib95.load(f"{file.absolute()}")
#    if problem.edge_weight_type != "EUC_2D": # Skip non-Euclidean TSPs
#        continue
#    if problem.dimension > SIZE_THRESHOLD2 or problem.dimension <= 350: # Skip large TSPs
#        continue
#    all_problems.append(problem)

#len(all_problems)

In [6]:
@njit(cache=True, inline="always", boundscheck=False, fastmath=True)
def wsp_heuristic_good(a_list: np.ndarray, b_list: np.ndarray, pos_in_tour: np.ndarray) -> bool:
    """A heuristic to check if a path is good based on the WSPs

    Takes in the positions of the nodes in A and B in the tour (sorted), and the total number of points in the problem.
    a_pos \subseteq [0, num_points-1] are the positions of the nodes in A in the tour
    b_pos \subseteq [0, num_points-1] are the positions of the nodes in B in the tour

    We then check the following conditions:
    1. If there are no exit pairs with endpoints in separate sets, then there must be exactly two edges connecting A and B
    2. If there are an even number of exit pairs with endpoints in separate sets, then there must be no edges connecting A and B 
    3. If there are an odd number of exit pairs with endpoints in separate sets, then there must be exactly one edge connecting A and B
    """
    num_points = len(pos_in_tour)
    #assert len(a_pos) > 0 and len(b_pos) > 0, "Sets must be non-empty"

    # keep track of how many edges directly connect A and B in the tour
    biconn_AB_count = 0
    biconn_BA_count = 0
    # keep track of how many exit pairs have endpoints in separate sets
    exit_AA_count = 0
    exit_BB_count = 0
    exit_AB_count = 0
    exit_BA_count = 0

    # a_pos[i] = pos_in_tour[a_list[i]]

    # check the edge cases
    if pos_in_tour[a_list[-1]] == num_points - 1 and pos_in_tour[b_list[0]] == 0:
        biconn_AB_count += 1
    elif pos_in_tour[b_list[-1]] == num_points - 1 and pos_in_tour[a_list[0]] == 0:
        biconn_BA_count += 1
    elif (pos_in_tour[a_list[-1]] == num_points - 1 and pos_in_tour[a_list[0]] == 0) or (pos_in_tour[b_list[-1]] == num_points - 1 and pos_in_tour[b_list[0]] == 0):
        pass # then there is just an a loop and nothing to see here
    elif pos_in_tour[a_list[-1]] > pos_in_tour[b_list[-1]]:
        if pos_in_tour[a_list[0]] < pos_in_tour[b_list[0]]:
            exit_AA_count += 1
        else:
            exit_AB_count += 1
    else:
        if pos_in_tour[b_list[0]] < pos_in_tour[a_list[0]]:
            exit_BB_count += 1
        else:
            exit_BA_count += 1

    # two pointer approach to count biconns and exits
    i = j = 0
    na = len(a_list)
    nb = len(b_list)

    # tracking info
    next_a = pos_in_tour[a_list[0]]
    next_b = pos_in_tour[b_list[0]]
    exited_A = None

    while True:
        if next_a <= next_b:
            # handle if exited
            if exited_A is not None:
                if exited_A:
                    exit_AA_count += 1
                else:
                    exit_BA_count += 1
                exited_A = None

            # check if biconn or exit
            if next_a + 1 == next_b:
                biconn_AB_count += 1
            elif i + 1 >= na or next_a + 1 != pos_in_tour[a_list[i+1]]:
                exited_A = True

            # increment i and update next_a
            i += 1
            if i < na:
                next_a = pos_in_tour[a_list[i]]
            else:
                break
        else:
            # handle if exited
            if exited_A is not None:
                if exited_A:
                    exit_AB_count += 1
                else:
                    exit_BB_count += 1
                exited_A = None

            # check if biconn or exit
            if next_b + 1 == next_a:
                biconn_BA_count += 1
            elif j + 1 >= nb or next_b + 1 != pos_in_tour[b_list[j+1]]:
                exited_A = False
            
            # increment j and update next_b
            j += 1
            if j < nb:
                next_b = pos_in_tour[b_list[j]]
            else:
                break
    # at this point one or both sets are expended
    while i < na:
        if exited_A is not None:
            if exited_A:
                exit_AA_count += 1
            else:
                exit_BA_count += 1
            exited_A = None
        if i + 1 >= na or pos_in_tour[a_list[i]] + 1 != pos_in_tour[a_list[i+1]]:
            exited_A = True
        i += 1
    while j < nb:
        if exited_A is not None:
            if exited_A:
                exit_AB_count += 1
            else:
                exit_BB_count += 1
            exited_A = None
        if j + 1 >= nb or pos_in_tour[b_list[j]] + 1 != pos_in_tour[b_list[j+1]]:
            exited_A = False
        j += 1
        

    ## covers both single in outs and multi case
    cross_exits = exit_AB_count + exit_BA_count
    if cross_exits == 0:
        return biconn_AB_count == 1 and biconn_BA_count == 1
    elif cross_exits % 2 == 0:
        return biconn_AB_count == 0 and biconn_BA_count == 0
    else:
        return (biconn_AB_count == 1 and biconn_BA_count == 0) or (biconn_AB_count == 0 and biconn_BA_count == 1)

In [ ]:
@nb.njit(parallel=True, boundscheck=False, fastmath=True)
def count_bad_wspd_parallel(pos_in_tour, a_list, b_list, a_offsets, b_offsets):
    #bad_count = 0
    num_pairs = len(a_offsets)
    is_bad = np.zeros(num_pairs, dtype=np.bool_)
    
    # Store total lengths for the final boundary check
    total_a_len = len(a_list)
    total_b_len = len(b_list)
    
    for i in nb.prange(num_pairs):
        a_start = a_offsets[i]
        b_start = b_offsets[i]
        
        # Safely determine the end boundaries
        if i < num_pairs - 1:
            a_end = a_offsets[i+1]
            b_end = b_offsets[i+1]
        else:
            # On the final pair, the end is just the end of the flat array
            a_end = total_a_len
            b_end = total_b_len
        
        # if both sets have more than 1 point, then we need to check the heuristic. Otherwise, it is automatically satisfied
        if (a_end - a_start) > 1 and (b_end - b_start) > 1:
            # 3. POSITION MAPPING & SORTING
            A = a_list[a_start:a_end]
            B = b_list[b_start:b_end]

            sort_by_key_inplace(A, pos_in_tour)
            sort_by_key_inplace(B, pos_in_tour)

            # 4. HEURISTIC CHECK
            if not wsp_heuristic_good(A, B, pos_in_tour):
                #bad_count += 1
                is_bad[i] = True

    return np.flatnonzero(is_bad)

def check_tour_with_wspd(wspd: tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray], tour: np.ndarray):
    a_list, b_list, a_offsets, b_offsets = wspd

    # vectorized node -> position map
    pos_in_tour = np.empty(len(tour), dtype=np.uint32)
    pos_in_tour[tour] = np.arange(len(tour), dtype=np.uint32)

    #biggest_pair = 0
    bad_count = 0

    # 2. Execute parallel loop natively in Numba
    bad_count = count_bad_wspd_parallel(
        pos_in_tour,
        a_list, 
        b_list, 
        a_offsets, 
        b_offsets,
    )

    return bad_count

def check_problem_tour(prob: tsplib95.models.StandardProblem, tour: np.ndarray, dim=2) -> tuple[int, int | None]:
    """Given a TSP and a non-wrapping tour, checks if the WSP heuristic holds"""
    points = [wsp_point(prob.node_coords[i]) for i in prob.get_nodes()]

    # Build the WSPDs with different separation factors
    wspd_15: tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray] = build_wspd_flat_np(len(points), dim, 1.5*2, points)
    print("built wspd1.5", flush=True)

    return len(check_tour_with_wspd(wspd_15, tour)), None

In [8]:
#def profile_bottleneck(all_problems: list[tsplib95.models.StandardProblem]):

#for prob in all_problems:
#    print(f"Checking {prob.name}...", flush=True)

#    lkh_instance = elkai.Coordinates2D({
#        i-1: prob.node_coords[i] for i in prob.get_nodes()
#    })

#    # tour numpy conversions
#    tour_lkh = lkh_instance.solve_tsp(runs=1)
#    tour = np.array(tour_lkh, dtype=np.uint16)

#    check_problem_tour(prob, tour)

#%lprun -f profile_bottleneck -u 1.0 profile_bottleneck(all_problems)
#print("Done")

## Loading previously solved TSPs

In [9]:
gaia_prob = tsplib95.load("ALL_tsp/mona-lisa100K.tsp")
gaia_tour_prob = tsplib95.load("ALL_tsp/mona-lisa100K.tour")

In [10]:
gaia_tour = np.array(gaia_tour_prob.tours[0], dtype=np.uint32) - 1
gaia_tour.shape

(100000,)

In [11]:
#from any_metric_wspd import gen_wspd
#from scipy.spatial.distance import pdist, squareform

#points = np.asarray([gaia_prob.node_coords[i] for i in gaia_prob.get_nodes()], dtype=np.float32)
#dist_matrix = pdist(points, metric="euclidean")
#dist_matrix = squareform(dist_matrix)
#dist_matrix.shape

#wspd = gen_wspd(dist_matrix, 0.9999)
#len(wspd)

In [14]:
check_problem_tour(gaia_prob, gaia_tour, dim=2)

#%lprun -f check_problem_tour -u 1.0 check_problem_tour(gaia_prob, gaia_tour, dim=2)

built wspd1.5


(0, None)

In [15]:
# Scan for all "*.opt.tour" files and test matching EUC_2D / EUC_3D instances
search_roots = [TSP_LIB_PATH, Path("ALL_tsp"), Path(".")]
opt_tour_files = []

seen = set()
for root in search_roots:
    if not root.exists():
        continue
    for p in root.rglob("*.opt.tour"):
        rp = p.resolve()
        if rp not in seen:
            seen.add(rp)
            opt_tour_files.append(p)

opt_tour_files = sorted(opt_tour_files)

results = []
failures = []

for opt_path in opt_tour_files:
    try:
        tsp_path = opt_path.with_suffix("").with_suffix(".tsp")  # *.opt.tour -> *.tsp
        if not tsp_path.exists():
            failures.append((str(opt_path), "matching .tsp not found"))
            continue

        prob = tsplib95.load(str(tsp_path))
        if prob.edge_weight_type not in ("EUC_2D", "EUC_3D"):
            continue

        tour_prob = tsplib95.load(str(opt_path))
        if not getattr(tour_prob, "tours", None):
            failures.append((prob.name, "no tours in opt file"))
            continue

        dim = 2 if prob.edge_weight_type == "EUC_2D" else 3

        # Robust node-id -> index mapping (does not assume nodes are exactly 1..n)
        nodes = list(prob.get_nodes())
        node_to_idx = {node_id: idx for idx, node_id in enumerate(nodes)}

        raw_tour = tour_prob.tours[0]
        try:
            tour_idx = np.array([node_to_idx[n] for n in raw_tour], dtype=np.uint32)
        except KeyError as e:
            failures.append((prob.name, f"tour contains unknown node id: {e}"))
            continue

        # Ensure wrapped/closed tour for check_problem_tour()
        if tour_idx[0] != tour_idx[-1]:
            tour_idx = np.concatenate([tour_idx, tour_idx[:1]])

        print(f"Checking {prob.name} ({prob.edge_weight_type}, n={prob.dimension})...", flush=True)
        bad_count, biggest_pair = check_problem_tour(prob, tour_idx, dim=dim)

        results.append({
            "name": prob.name,
            "type": prob.edge_weight_type,
            "n": prob.dimension,
            "bad_pairs": bad_count,
            "biggest_pair": biggest_pair,
            "opt_tour_file": str(opt_path),
        })

    except Exception as e:
        failures.append((str(opt_path), repr(e)))

print(f"\nChecked {len(results)} EUC_2D/EUC_3D problems with *.opt.tour")
print(f"Failures: {len(failures)}")
if failures:
    print("First 10 failures:")
    for item in failures[:10]:
        print(" -", item)

# Quick aggregate summary
if results:
    total_bad = sum(r["bad_pairs"] for r in results)
    bad_instances = sum(1 for r in results if r["bad_pairs"] > 0)
    print(f"\nTotal bad pairs: {total_bad}")
    print(f"Instances with >=1 bad pair: {bad_instances}/{len(results)}")

Checking Tnm100.tsp (EUC_2D, n=100)...
built wspd1.5
Checking Tnm103.tsp (EUC_2D, n=103)...
built wspd1.5
Checking Tnm106.tsp (EUC_2D, n=106)...
built wspd1.5
Checking Tnm109.tsp (EUC_2D, n=109)...
built wspd1.5
Checking Tnm112.tsp (EUC_2D, n=112)...
built wspd1.5
Checking Tnm115.tsp (EUC_2D, n=115)...
built wspd1.5
Checking Tnm118.tsp (EUC_2D, n=118)...
built wspd1.5
Checking Tnm121.tsp (EUC_2D, n=121)...
built wspd1.5
Checking Tnm124.tsp (EUC_2D, n=124)...
built wspd1.5
Checking Tnm127.tsp (EUC_2D, n=127)...
built wspd1.5
Checking Tnm130.tsp (EUC_2D, n=130)...
built wspd1.5
Checking Tnm133.tsp (EUC_2D, n=133)...
built wspd1.5
Checking Tnm136.tsp (EUC_2D, n=136)...
built wspd1.5
Checking Tnm139.tsp (EUC_2D, n=139)...
built wspd1.5
Checking Tnm142.tsp (EUC_2D, n=142)...
built wspd1.5
Checking Tnm145.tsp (EUC_2D, n=145)...
built wspd1.5
Checking Tnm148.tsp (EUC_2D, n=148)...
built wspd1.5
Checking Tnm151.tsp (EUC_2D, n=151)...
built wspd1.5
Checking Tnm154.tsp (EUC_2D, n=154)...
built w

## Checking large instances

In [16]:
gaia_prob = tsplib95.load("ALL_tsp/gaia2079471.tsp") # gaia2079471
gaia_tour_prob = tsplib95.load("ALL_tsp/gaia2079471.tour")
gaia_tour = np.array(gaia_tour_prob.tours[0], dtype=np.uint32) - 1

In [17]:
points = [wsp_point(gaia_prob.node_coords[i]) for i in gaia_prob.get_nodes()]

# Build the WSPDs with different separation factors
wspd_15: tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray] = build_wspd_flat_np(len(points), 3, 1.5*2, points)

In [18]:
wspd_15[1].shape

(3865497465,)

In [19]:
bad_pair_inds = check_tour_with_wspd(wspd_15, gaia_tour)